# Dim_Year - Gold

Unique years from regional appropriations.

## 1. Create the dimension table

- Keep distinct, non-null years from the Silver table.
- Assign `year_id` values starting at `20000`, ordered by year.
- Store one row per year.


In [0]:
%sql
CREATE TABLE IF NOT EXISTS pondolens.dev_gold.dim_year
USING DELTA
AS
WITH distinct_years AS (
  SELECT DISTINCT year
  FROM pondolens.dev_silver.regional_appropriations
  WHERE year IS NOT NULL
),
id_start AS (
  SELECT 19999 AS starting_id
)
SELECT
  CAST(
    b.starting_id + ROW_NUMBER() OVER (ORDER BY y.year)
    AS INT
  ) AS year_id,
  y.year
FROM distinct_years AS y
CROSS JOIN id_start AS b;

## 2. View the dimension

Display the years and their assigned IDs.

In [0]:
%sql
SELECT *
FROM pondolens.dev_gold.dim_year

## 3. Data quality checks

Check for missing values, duplicate IDs or years, incorrect column types, and differences from the source.

- **PASS:** No issues found.
- **REVIEW:** More than 0% up to 5% affected.
- **FAIL:** More than 5% affected.
- Checks with nothing to evaluate return **REVIEW**.


In [0]:
%sql
-- Read-only report. PASS: 0%; REVIEW: >0% to 5%; FAIL: >5%.
WITH settings AS (
  SELECT 5.0 AS review_max_pct
),
source_years AS (
  SELECT DISTINCT year
  FROM pondolens.dev_silver.regional_appropriations
  WHERE year IS NOT NULL
),
dim AS (
  SELECT *
  FROM pondolens.dev_gold.dim_year
),
totals AS (
  SELECT
    (SELECT COUNT(*) FROM dim) AS dim_rows,
    (SELECT COUNT(*) FROM source_years) AS source_year_count
),
duplicate_ids AS (
  SELECT year_id
  FROM dim
  WHERE year_id IS NOT NULL
  GROUP BY year_id
  HAVING COUNT(*) > 1
),
duplicate_years AS (
  SELECT year
  FROM dim
  WHERE year IS NOT NULL
  GROUP BY year
  HAVING COUNT(*) > 1
),
checks AS (
  SELECT
    1 AS sort_order,
    'Missing year IDs or years' AS dq_check,
    COUNT(*) AS affected_count,
    'dimension rows' AS count_basis
  FROM dim
  WHERE year_id IS NULL OR year IS NULL

  UNION ALL

  -- Count every row belonging to a duplicate group.
  SELECT
    2,
    'Duplicate year IDs',
    COUNT(*),
    'dimension rows'
  FROM dim AS d
  LEFT SEMI JOIN duplicate_ids AS x
    ON d.year_id = x.year_id

  UNION ALL

  SELECT
    3,
    'Duplicate years',
    COUNT(*),
    'dimension rows'
  FROM dim AS d
  LEFT SEMI JOIN duplicate_years AS x
    ON d.year = x.year

  UNION ALL

  SELECT
    4,
    'IDs below 20000',
    COUNT(*),
    'dimension rows'
  FROM dim
  WHERE year_id < 20000

  UNION ALL

  SELECT
    5,
    'Source years missing from dimension',
    COUNT(*),
    'source years'
  FROM source_years AS s
  LEFT ANTI JOIN dim AS d
    ON s.year = d.year

  UNION ALL

  SELECT
    6,
    'Dimension years absent from source',
    COUNT(*),
    'dimension rows'
  FROM dim AS d
  LEFT ANTI JOIN source_years AS s
    ON d.year = s.year
  WHERE d.year IS NOT NULL

  UNION ALL

  SELECT
    7,
    'Incorrect or missing column types',
    2 - COUNT(*),
    'columns'
  FROM pondolens.information_schema.columns
  WHERE table_schema = 'dev_gold'
    AND table_name = 'dim_year'
    AND column_name IN ('year_id', 'year')
    AND UPPER(data_type) = 'INT'
),
evaluated AS (
  SELECT
    c.*,
    CASE c.count_basis
      WHEN 'source years' THEN t.source_year_count
      WHEN 'columns' THEN 2
      ELSE t.dim_rows
    END AS total_checked
  FROM checks AS c
  CROSS JOIN totals AS t
),
percentages AS (
  SELECT
    *,
    100.0 * affected_count / NULLIF(total_checked, 0)
      AS affected_pct
  FROM evaluated
)
SELECT
  p.dq_check,
  p.total_checked,
  p.affected_count,
  ROUND(p.affected_pct, 2) AS affected_percentage,
  -- Assign status using the unrounded percentage.
  CASE
    WHEN p.total_checked = 0 THEN 'REVIEW'
    WHEN p.affected_count = 0 THEN 'PASS'
    WHEN p.affected_pct <= s.review_max_pct THEN 'REVIEW'
    ELSE 'FAIL'
  END AS status
FROM percentages AS p
CROSS JOIN settings AS s
ORDER BY p.sort_order;

## 4. Data Quality Results

- **5 years:** 2021–2025, with IDs from **20000 to 20004**.
- All **7 quality checks passed**, with zero issues.
- No missing values, duplicate IDs, or duplicate years were found.
- Dimension years match the source, with no missing or extra years.
- Both columns have the expected `INT` data type.